In [7]:
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "browser"

def f(x, y, z):
    return (x**2)/2 + (y**2)/3 + (z**2)/4 - 15


grid_size = 18

x = np.linspace(-10, 10, grid_size)
y = np.linspace(-10, 10, grid_size)
z = np.linspace(-10, 10, grid_size)

Xg, Yg, Zg = np.meshgrid(x, y, z, indexing="ij")
Fg = f(Xg, Yg, Zg)

cube_edges = []
cube_size = (x[1] - x[0])
rng = np.random.default_rng(0)

def make_cube_lines(x0,y0, z0, size):

    dx = [0, size]
    lines = []

    for sx in dx:
        for sy in dx:
            lines.append(([x0+sx,x0+sx],[y0+sy,y0+sy],[z0,z0+size]))

    for sx in dx:
        for sz in dx:
            lines.append(([x0+sx,x0+sx],[y0,y0+size],[z0+sz,z0+sz]))

    for sy in dx:
        for sz in dx:
            lines.append(([x0,x0+size],[y0+sy,y0+sy],[z0+sz,z0+sz]))

    return lines


for i in range(grid_size-1):
    for j in range(grid_size-1):
        for k in range(grid_size-1):

            vals = Fg[i:i+2,j:j+2,k:k+2]

            if np.min(vals)*np.max(vals) < 0:

                if rng.random() < 0.8:

                    cube_edges.extend(
                        make_cube_lines(x[i],y[j],z[k],cube_size)
                    )

a = np.sqrt(2 * 15)
b = np.sqrt(3 * 15)
c = np.sqrt(4 * 15)

n_theta = 80
n_phi = 50

theta = np.linspace(0, 2*np.pi, n_theta, endpoint=False)
phi = np.linspace(0, np.pi, n_phi)

Theta, Phi = np.meshgrid(theta, phi)

X_s = a*np.sin(Phi)*np.cos(Theta)
Y_s = b*np.sin(Phi)*np.sin(Theta)
Z_s = c*np.cos(Phi)

X_flat = X_s.ravel()
Y_flat = Y_s.ravel()
Z_flat = Z_s.ravel()


from scipy.spatial import Delaunay
points_2d = np.vstack([Theta.ravel(), Phi.ravel()]).T
tri = Delaunay(points_2d)

faces = tri.simplices

fig1 = go.Figure()

for lx,ly,lz in cube_edges:

    fig1.add_trace(go.Scatter3d(
        x=lx,
        y=ly,
        z=lz,
        mode="lines",
        line=dict(color="rgba(130,130,130,0.75)",width=2),
        showlegend=False
    ))

fig1.add_trace(go.Scatter3d(

x=X_flat,
y=Y_flat,
z=Z_flat,

mode="markers",

marker=dict(
color="#09122C",
size=2
)

))

fig1.update_layout(

title=f"ՔԱՅԼ1 Մակերևույթի կետեր (քանակը = {len(X_flat)})",

scene=dict(
xaxis_title="X",
yaxis_title="Y",
zaxis_title="Z"
)

)

fig1.show()


fig_partial = go.Figure()

fig_partial.add_trace(go.Scatter3d(

    x=X_flat,
    y=Y_flat,
    z=Z_flat,

    mode="markers",

    marker=dict(
        size=2,
        color="#09122C"
    ),

    showlegend=False
))


n_30 = int(0.3 * len(faces))

for face in faces[:n_30]:

    for i in range(3):

        p1 = face[i]
        p2 = face[(i+1)%3]

        fig_partial.add_trace(go.Scatter3d(

            x=[X_flat[p1], X_flat[p2]],
            y=[Y_flat[p1], Y_flat[p2]],
            z=[Z_flat[p1], Z_flat[p2]],

            mode="lines",

            line=dict(
                color="#BE3144",
                width=2
            ),

            showlegend=False
        ))


fig_partial.update_layout(

title=f"ՔԱՅԼ 1.5 — ~30% Դելոնեի եռանկյունացում",

width=1000,
height=800,

scene=dict(
aspectmode="data",
xaxis_title="X",
yaxis_title="Y",
zaxis_title="Z"
)

)

fig_partial.show()

fig2 = go.Figure()

for lx,ly,lz in cube_edges:

    fig2.add_trace(go.Scatter3d(
        x=lx,
        y=ly,
        z=lz,
        mode="lines",
        line=dict(color="rgba(130,130,130,0.75)",width=2),
        showlegend=False
    ))

tech_gradient = [
[0.0,"#09122C"],
[0.25,"#09122C"],
[0.5,"#872341"],
[0.75,"#BE3144"],
[1.0,"#E17564"]
]

fig2.add_trace(go.Mesh3d(
    x=X_flat,
    y=Y_flat,
    z=Z_flat,
    
    alphahull=0, 
    
    intensity=Z_flat,
    colorscale=tech_gradient,
    opacity=1
))

fig2.update_layout(

title=f"ՔԱՅԼ 2 — Եռանկյունացված մակերևույթ (եռանկյուններ = {len(faces)})",

scene=dict(
xaxis_title="X",
yaxis_title="Y",
zaxis_title="Z"
)

)

fig2.show()


print("Grid կետերի քանակ =", Xg.size)
print("Մակերևույթի կետերի քանակ =", len(X_flat))
print("Եռանկյունների քանակ =", len(faces))

Grid կետերի քանակ = 5832
Մակերևույթի կետերի քանակ = 4000
Եռանկյունների քանակ = 7742
